# RepoGuard — Phase 2: Overnight RL Training & Evaluation on Google Colab

> **Project:** RepoGuard (Final Year Engineering Project)  
> **Workstream:** Member 3 — RL Agent Development, Training, Evaluation & Cloud Pipeline  
> **Repository:** [https://github.com/Shreyank86/Repo-Guard](https://github.com/Shreyank86/Repo-Guard)  
> **Key Objective:** Full-scale 2,000-episode DQN & 1,500-episode REINFORCE training on GPU with **automatic checkpoint resume**, **persistent Drive caching & indexes**, and **held-out test split evaluation**.

---

### Architectural Invariants Preserved:
- **Frozen Model:** `Qwen/Qwen2.5-Coder-3B-Instruct` (with 4-bit NF4 quantization on CUDA).
- **State Space:** 18-dimensional normalized state vector ($s \in [0, 1]^{18}$).
- **Action Space:** Canonical 6 actions (`SEARCH`, `INSPECT`, `VERIFY`, `ANSWER`, `ANSWER_WITH_EVIDENCE`, `ABSTAIN`).
- **Dynamic Risk Penalty:** $\lambda \in [0.25, 5.0]$ sampled per episode.
- **Reward Formula:** $r = R_{\text{final}}(\lambda) - 0.2 \cdot \frac{T}{2000} - 0.01 \cdot n$.
- **Fault Tolerance:** Checkpoints, operation cache (`operations.db`), and vector indexes are persisted directly to **Google Drive**.

## 1. Verify GPU Hardware
Verify that Google Colab has allocated an NVIDIA GPU (T4, V100, or A100).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive & Setup Persistent Directories
Mount Google Drive so that indexes, cached LLM outputs, checkpoints, and reports persist permanently even if the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/RepoGuard')
DRIVE_INDEXES = DRIVE_ROOT / 'data' / 'indexes'
DRIVE_CACHE = DRIVE_ROOT / 'data' / 'cache'
DRIVE_CHECKPOINTS_DQN = DRIVE_ROOT / 'checkpoints' / 'dqn'
DRIVE_CHECKPOINTS_RF = DRIVE_ROOT / 'checkpoints' / 'reinforce'
DRIVE_REPORTS = DRIVE_ROOT / 'reports'

for d in [DRIVE_INDEXES, DRIVE_CACHE, DRIVE_CHECKPOINTS_DQN, DRIVE_CHECKPOINTS_RF, DRIVE_REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Persistent Drive workspace ready at: {DRIVE_ROOT}")

## 3. Clone Repository & Verify Commit
Clone the RepoGuard GitHub repository (or pull latest changes if already present).

In [ ]:
%cd /content
REPO_URL = "https://github.com/Shreyank86/Repo-Guard.git"

if not Path('/content/Repo-Guard').exists():
    !git clone {REPO_URL} /content/Repo-Guard
else:
    %cd /content/Repo-Guard
    !git pull origin main

%cd /content/Repo-Guard
!git log -1 --format="Latest commit: %h - %s (%ci)"

## 4. Install Dependencies
Install project requirements, `bitsandbytes` (for 4-bit NF4 quantized inference), and the `repoguard` package in editable mode.

In [ ]:
%cd /content/Repo-Guard
!pip install -q -r requirements.txt
!pip install -q bitsandbytes
!pip install -q -e .
print("All project dependencies installed successfully!")

## 5. Persistent Repository & Vector Index Setup
Clones the 5 pinned benchmark repositories (`schedule`, `python-dotenv`, `cachetools`, `python-tabulate`, `requests`).  
- Reuses pre-built indexes from Google Drive if they exist.  
- Builds and backs up indexes to Google Drive if missing so they are never rebuilt twice.

In [ ]:
%cd /content/Repo-Guard
import os
import shutil
from pathlib import Path

# 1. Fetch source code for the 5 pinned repositories
!python scripts/fetch_repos.py

# 2. Check if indexes are already cached in Google Drive
required_repos = ["schedule", "python-dotenv", "cachetools", "python-tabulate", "requests"]
drive_has_all_indexes = all((DRIVE_INDEXES / name / "metadata.json").exists() for name in required_repos)

local_idx_dir = Path("/content/Repo-Guard/data/indexes")
local_idx_dir.parent.mkdir(parents=True, exist_ok=True)

if drive_has_all_indexes:
    print("Reusing pre-computed indexes from Google Drive (skipping rebuild)...")
    if local_idx_dir.is_symlink():
        local_idx_dir.unlink()
    elif local_idx_dir.exists():
        shutil.rmtree(local_idx_dir, ignore_errors=True)
    os.symlink(str(DRIVE_INDEXES), str(local_idx_dir))
    print(f"Symlinked {DRIVE_INDEXES} -> {local_idx_dir}")
else:
    print("Building indexes and persisting them to Google Drive...")
    !python scripts/fetch_repos.py --index
    !cp -r /content/Repo-Guard/data/indexes/* /content/drive/MyDrive/RepoGuard/data/indexes/
    print("Indexes built and persisted to Google Drive!")

## 6. Unit Tests Smoke Check
Verify the RL environment, action executor, DQN agent, REINFORCE agent, and policy serving wrappers.

In [ ]:
%cd /content/Repo-Guard
!pytest tests/unit/test_agent_cache.py tests/unit/test_environment.py tests/unit/test_dqn.py tests/unit/test_reinforce.py tests/unit/test_policy_serving.py -v

## 7. Pre-Training Pre-Flight Validation
Strict validation gate verifying:  
1. All 5 repositories exist and match expected pinned commit hashes.  
2. All 5 vector indexes exist, load with Schema 1.1, and have non-zero symbols and edges.  
3. Benchmark dataset exists and has valid `train` (36), `validation` (12), and `test` (12) splits.  
4. 18-dim observation space and 6 discrete actions are intact.  
5. SQLite operation cache on Google Drive is writable.  
6. GPU and VRAM are operational.

In [ ]:
%cd /content/Repo-Guard
import yaml
import subprocess
from pathlib import Path
from repoguard.repository.indexer import load_index
from repoguard.evaluation.benchmark import load_benchmark
from repoguard.agent.cache import OperationCache
from repoguard.agent.environment import RepoGuardEnv, ACTION_LOOKUP
from repoguard.llm import LLMService
from repoguard.schemas import RLAction

print("=== PRE-TRAINING VALIDATION GATE ===")

# 1. Verify Repositories & Commits
cfg = yaml.safe_load(Path("configs/repos.yaml").read_text())["repositories"]
for r in cfg:
    name, commit = r["name"], r["commit"]
    repo_path = Path("data/repos") / name
    assert repo_path.exists(), f"Missing repository: {name}"
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_path, capture_output=True, text=True, check=True).stdout.strip()
    assert head == commit, f"Repo {name} HEAD {head} != pinned {commit}"
    print(f"  [OK] Repo {name:16s} pinned at {commit[:8]}")

# 2. Verify Vector Indexes
for r in cfg:
    idx = load_index(Path("data/indexes") / r["name"])
    assert idx.schema_version == "1.1", f"Invalid schema version: {idx.schema_version}"
    assert len(idx.symbols) > 0, f"Zero symbols in {r['name']}"
    assert len(idx.passages) > 0, f"Zero passages in {r['name']}"
    print(f"  [OK] Index {r['name']:15s} schema 1.1 | {len(idx.symbols)} symbols | {len(idx.passages)} passages")

# 3. Verify Benchmark Splits
cases = load_benchmark(Path("data/questions/phase2_benchmark_v1.jsonl"))
splits = {s: sum(1 for c in cases if getattr(c, "split", "") == s) for s in ["train", "validation", "test"]}
assert splits["train"] == 36, f"Expected 36 train cases, got {splits['train']}"
assert splits["validation"] == 12, f"Expected 12 validation cases, got {splits['validation']}"
assert splits["test"] == 12, f"Expected 12 test cases, got {splits['test']}"
print(f"  [OK] Benchmark: 60 cases (Train: {splits['train']}, Val: {splits['validation']}, Test: {splits['test']})")

# 4. Verify 18-dim State and 6-Action Contracts
assert len(ACTION_LOOKUP) == 6, f"Expected 6 canonical actions, got {len(ACTION_LOOKUP)}"
llm = LLMService(mock_mode=True)
indexes = {r["name"]: load_index(Path("data/indexes") / r["name"]) for r in cfg}
env = RepoGuardEnv(benchmark_cases=cases, indexes=indexes, llm_service=llm)
obs, info = env.reset(case_index=0)
assert obs.shape == (18,), f"Expected 18-dim observation vector, got {obs.shape}"
assert len(info["valid_mask"]) == 6, f"Expected 6-dim action mask, got {len(info['valid_mask'])}"
print("  [OK] Action Space: 6 discrete actions | Observation Space: 18-dim normalized Box")

# 5. Verify SQLite Cache on Drive
cache_file = Path('/content/drive/MyDrive/RepoGuard/data/cache/operations.db')
cache = OperationCache(db_path=cache_file)
cache.set("TEST_OP", "schedule", "0b7774c", {"test": 1}, {"status": "ok"})
assert cache.has("TEST_OP", "schedule", "0b7774c", {"test": 1})
print(f"  [OK] Persistent SQLite Cache accessible and writable on Drive: {cache_file}")

print("\n>>> ALL PRE-TRAINING CHECKS PASSED! SYSTEM READY FOR OVERNIGHT TRAINING <<<")

## 8. Offline Transitions & Operation Cache Status
- `data/episodes/episode_log.jsonl` contains pre-recorded transitions collected across the benchmark.
- `train_agent.py` executes **online environment interaction** (`step()`) while leveraging `OperationCache` so identical operations are resolved in <1ms without re-querying models.

In [ ]:
%cd /content/Repo-Guard
import json
from collections import Counter
from pathlib import Path
from repoguard.agent.cache import OperationCache

ep_file = Path("data/episodes/episode_log.jsonl")
if not ep_file.exists():
    print("Pre-populating offline transitions via run_episodes.py...")
    !python scripts/run_episodes.py --output data/episodes/episode_log.jsonl

records = [json.loads(line) for line in ep_file.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Loaded {len(records)} transitions from {ep_file}")
assert len(records) > 0, "Episode log cannot be empty!"

# Validate schema and value constraints on every transition
required_keys = {"case_id", "repository_id", "state", "action", "reward", "next_state", "done", "valid_mask"}
for idx, r in enumerate(records):
    missing = required_keys - set(r.keys())
    assert not missing, f"Record {idx} missing fields: {missing}"
    assert len(r["state"]) == 18, f"Record {idx} state length != 18"
    assert len(r["next_state"]) == 18, f"Record {idx} next_state length != 18"
    assert len(r["valid_mask"]) == 6, f"Record {idx} valid_mask length != 6"

cases_covered = len(set(r["case_id"] for r in records))
repos_covered = len(set(r["repository_id"] for r in records))
action_dist = Counter(r.get("action_name", str(r["action"])) for r in records)

print(f"  [OK] Validated {len(records)} transitions across {cases_covered} cases and {repos_covered} repositories.")
print(f"  [OK] Action distribution: {dict(action_dist)}")

# Check Operation Cache size in Google Drive
cache_file = Path('/content/drive/MyDrive/RepoGuard/data/cache/operations.db')
cache = OperationCache(db_path=cache_file)
print(f"  [OK] Operation Cache entries in Drive: {cache.size()}")

## 9. 30-Second Smoke Test: Save & Auto-Resume Verification
Run a fast 5-episode smoke test to verify checkpoint saving and resume recovery on Colab GPU before launching the overnight run.

In [ ]:
%cd /content/Repo-Guard

print("=== SMOKE TEST: 5 EPISODES + RESUME VERIFICATION ===")
# 1. Run 5 episodes of DQN to a smoke_test checkpoint dir
!python scripts/train_agent.py \
    --algo dqn \
    --episodes 5 \
    --device cuda \
    --seed 42 \
    --save-interval 2 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/smoke_test \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --log-interval 1

# 2. Re-run targeting 10 episodes to confirm auto-resume picks up at episode 6
!python scripts/train_agent.py \
    --algo dqn \
    --episodes 10 \
    --device cuda \
    --seed 42 \
    --auto-resume \
    --save-interval 2 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/smoke_test \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --log-interval 1

print("\n>>> SMOKE TEST PASSED! Auto-resume and Drive persistence confirmed operational. <<<")

## 10. Train Deep Q-Network (DQN) Policy (Full Overnight Run)

> [!TIP]
> **Overnight Execution:** `--auto-resume` is enabled. If Google Colab disconnects, simply re-run this cell; it automatically detects the latest checkpoint on Google Drive and continues seamlessly from where it left off without restarting.

**Hyperparameters:**
- Episodes: 2,000
- Seed: 42 (explicit for reproducibility)
- Exploration: $\epsilon: 1.0 \to 0.05$ (decay 0.995)
- Risk tolerance: $\lambda \sim \text{LogUniform}(0.25, 5.0)$
- Experience Replay: 20,000 transitions with action masking
- Checkpoints: Atomic save to Google Drive every 50 episodes

In [ ]:
%cd /content/Repo-Guard

!python scripts/train_agent.py \
    --algo dqn \
    --episodes 2000 \
    --device cuda \
    --seed 42 \
    --auto-resume \
    --learning-rate 0.0005 \
    --batch-size 32 \
    --save-interval 50 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/dqn \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --log-interval 50

## 11. Train REINFORCE Agent (Policy Gradient Baseline)
Train the REINFORCE baseline over 1,500 episodes with running-mean baseline subtraction.

In [ ]:
%cd /content/Repo-Guard

!python scripts/train_agent.py \
    --algo reinforce \
    --episodes 1500 \
    --device cuda \
    --seed 42 \
    --auto-resume \
    --learning-rate 0.0005 \
    --save-interval 50 \
    --checkpoint-dir /content/drive/MyDrive/RepoGuard/checkpoints/reinforce \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --log-interval 50

## 12. Validation Split Evaluation (Model Selection)
Evaluate both trained policies on the **validation split** (12 cases) across the 6 risk levels $\lambda \in \{0.25, 0.5, 1.0, 2.0, 3.0, 5.0\}$.

In [ ]:
%cd /content/Repo-Guard

# Evaluate DQN on Validation Split
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt \
    --split validation \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/val_dqn.csv

# Evaluate REINFORCE on Validation Split
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt \
    --split validation \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/val_reinforce.csv

print("Validation evaluation complete!")

## 13. Final Held-Out Test Evaluation
Evaluate both policies on the strictly **held-out test split** (12 cases) across all 6 risk levels.  
Outputs are saved to separate CSV files on Google Drive for academic reporting.

In [ ]:
%cd /content/Repo-Guard

# Evaluate DQN on Test Split
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt \
    --split test \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/test_dqn.csv

# Evaluate REINFORCE on Test Split
!python scripts/evaluate_rl.py \
    --policy /content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt \
    --split test \
    --cache-db /content/drive/MyDrive/RepoGuard/data/cache/operations.db \
    --output-csv /content/drive/MyDrive/RepoGuard/reports/test_reinforce.csv

print("Final held-out test evaluation complete!")

## 14. Comparative Risk Curves & Performance Analysis
Load final test CSVs and plot the 3 core risk trade-off curves:  
1. **Coverage vs. $\lambda$**: Demonstrates selective answering as risk penalty increases.  
2. **Hallucination Rate vs. $\lambda$**: Demonstrates suppression of unsupported claims.  
3. **Trap Abstention vs. $\lambda$**: Demonstrates proactive refusal on unanswerable questions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dqn_test = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/test_dqn.csv')
rf_test = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/test_reinforce.csv')

print("=== HELD-OUT TEST RESULTS: DQN ===")
display(dqn_test)

print("\n=== HELD-OUT TEST RESULTS: REINFORCE ===")
display(rf_test)

# Plot Publication-Quality Risk Curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Coverage
axes[0].plot(dqn_test['lambda'], dqn_test['coverage_pct'], 'o-', label='DQN (Ours)', color='royalblue', linewidth=2)
axes[0].plot(rf_test['lambda'], rf_test['coverage_pct'], 's--', label='REINFORCE', color='darkorange', linewidth=2)
axes[0].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[0].set_ylabel('Coverage (%)', fontsize=11)
axes[0].set_title('Answer Coverage vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. Hallucination Rate
axes[1].plot(dqn_test['lambda'], dqn_test['hallucination_rate_pct'], 'o-', label='DQN (Ours)', color='crimson', linewidth=2)
axes[1].plot(rf_test['lambda'], rf_test['hallucination_rate_pct'], 's--', label='REINFORCE', color='purple', linewidth=2)
axes[1].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[1].set_ylabel('Hallucination Rate (%)', fontsize=11)
axes[1].set_title('Hallucination Rate vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 3. Correct Abstention on Traps
axes[2].plot(dqn_test['lambda'], dqn_test['correct_abstention_pct'], 'o-', label='DQN (Ours)', color='forestgreen', linewidth=2)
axes[2].plot(rf_test['lambda'], rf_test['correct_abstention_pct'], 's--', label='REINFORCE', color='teal', linewidth=2)
axes[2].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[2].set_ylabel('Trap Abstention (%)', fontsize=11)
axes[2].set_title('Correct Trap Abstention vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
chart_path = '/content/drive/MyDrive/RepoGuard/reports/test_risk_curves.png'
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"Risk curves figure saved to: {chart_path}")

## 15. Experiment Summary & Manifest
Generate an experiment manifest recording git hashes, seeds, training durations, and artifact locations on Google Drive.

In [ ]:
import json
import subprocess
from pathlib import Path
import datetime as dt

git_hash = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

manifest = {
    "project": "RepoGuard",
    "phase": "Phase 2 - Member 3",
    "timestamp": dt.datetime.now().isoformat(),
    "git_commit": git_hash,
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "benchmark_cases": 60,
    "dqn_episodes": 2000,
    "reinforce_episodes": 1500,
    "artifacts": {
        "dqn_checkpoint": "/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt",
        "reinforce_checkpoint": "/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt",
        "val_evaluation": "/content/drive/MyDrive/RepoGuard/reports/val_dqn.csv",
        "test_evaluation": "/content/drive/MyDrive/RepoGuard/reports/test_dqn.csv",
        "risk_figure": "/content/drive/MyDrive/RepoGuard/reports/test_risk_curves.png",
    }
}

manifest_path = Path('/content/drive/MyDrive/RepoGuard/reports/experiment_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print("=== EXPERIMENT MANIFEST ===")
print(json.dumps(manifest, indent=2))
print(f"\nManifest persisted to: {manifest_path}")